# Public Transport Delay Analysis

**Team Members**

| Name | Email | Student ID |
|---|---|---|
| Sankalp Khira | skhira@stevens.edu | 20023909 |
| Rakshita Singh | rsingh39@stevens.edu | 20024023 |
| Pal Sanjaybhai Anghan | panghan@stevens.edu | 2004515 |

---

This notebook analyzes a dataset of 2,000 public-transport trips to understand delay patterns
and predict whether a trip will be delayed. We explore how factors like time of day, transport
type, traffic congestion, and holidays affect departure delays across buses, trams, trains,
and metro lines.

## 1. Imports & Setup

In [ ]:
import os, sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# Make sure the project modules are on the path
# (assumes the .py files are in the same directory as this notebook)
from data_loader import load_data, clean_data, data_summary
from analytics import (
    compute_statistics, compute_rms_delay, compute_percentiles,
    get_high_delays, delays_by_group, delay_generator,
    correlation_matrix, peak_hour_delay_summary, traffic_delay_summary,
)
from delay_predictor import RuleBasedPredictor, LogisticDelayPredictor
from visualizations import generate_all_plots

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

DATA_FILE = "public_transport_delays.csv"
THRESHOLD = 5.0

## 2. Data Loading & Cleaning

In [ ]:
raw = load_data(DATA_FILE)
print(f"Raw rows: {len(raw)}")
raw.head()

In [ ]:
df = clean_data(raw)
print(f"After cleaning: {len(df)}")
df.head()

In [ ]:
info = data_summary(df)
print(f"Date range      : {info['date_range'][0]}  to  {info['date_range'][1]}")
print(f"Transport types : {', '.join(info['transport_types'])}")
print(f"Unique routes   : {info['unique_routes']}")
print(f"Avg delay       : {info['avg_delay']} min")
print(f"% delayed       : {info['delayed_pct']}%")

## 3. Descriptive Statistics

In [ ]:
delays = list(delay_generator(df))
mean, std, median = compute_statistics(delays)
rms = compute_rms_delay(delays)
pcts = compute_percentiles(delays)

print(f"Mean   : {mean:.2f} min")
print(f"Std    : {std:.2f} min")
print(f"Median : {median:.1f} min")
print(f"RMS    : {rms:.2f} min")
print(f"Percentiles: {pcts}")

high = get_high_delays(delays, 10)
print(f"\nTrips > 10 min delay: {len(high)}  ({len(high)/len(delays)*100:.1f}%)")

## 4. Group Analysis

### 4.1 By Transport Type

In [ ]:
for transport, avg in sorted(delays_by_group(df, "transport_type").items()):
    print(f"  {transport:<8} {avg:.2f} min")

### 4.2 Peak vs Off-Peak

In [ ]:
peak_df = peak_hour_delay_summary(df)
peak_df

### 4.3 By Traffic Congestion

In [ ]:
traffic_df = traffic_delay_summary(df)
traffic_df

### 4.4 By Delay Category

In [ ]:
for cat, avg in sorted(delays_by_group(df, "delay_category").items()):
    print(f"  {cat:<10} {avg:.2f} min")

## 5. Rule-Based Prediction

In [ ]:
rule_pred = RuleBasedPredictor(threshold=THRESHOLD)
rule_pred.fit(df)

risky = rule_pred.high_risk_routes()
print(f"Threshold       : {THRESHOLD} min")
print(f"Total routes    : {len(rule_pred.network)}")
print(f"High-risk routes: {len(risky)}")

In [ ]:
# Top 10 routes by average delay (uses enumerate)
summary = rule_pred.route_summary()
top10 = sorted(summary.items(), key=lambda kv: kv[1]["avg_delay"], reverse=True)[:10]

print("Top 10 Routes by Avg Delay")
print("-" * 55)
for i, (route_id, info) in enumerate(top10, start=1):
    flag = "!!" if info["high_risk"] else "  "
    print(f"  {i:>3}. {flag} {route_id}  avg={info['avg_delay']:.1f}  "
          f"max={info['max_delay']}  trips={info['trips']}")

### 5.1 Set Operations on Delay Categories

In [ ]:
all_cats = rule_pred.network.all_delay_categories()
print(f"All delay categories across network: {all_cats}")

ids = list(rule_pred.network.route_ids)
if len(ids) >= 2:
    common = rule_pred.network.common_categories(ids[0], ids[1])
    print(f"Categories shared by {ids[0]} & {ids[1]}: {common}")
    unique = rule_pred.network.unique_categories(ids[0])
    print(f"Categories unique to {ids[0]}: {unique if unique else 'none (all shared)'}")

## 6. Logistic Regression Model

In [ ]:
lr = LogisticDelayPredictor()
results = lr.train(df)

print(f"Train size : {results['train_size']}")
print(f"Test size  : {results['test_size']}")
print(f"Accuracy   : {results['accuracy']:.2%}")
print()
print(results['report'])

In [ ]:
importance = lr.feature_importance()
print("Feature Importance (absolute coefficient)")
print("-" * 45)
for feat, coef in sorted(importance.items(), key=lambda x: x[1], reverse=True):
    print(f"  {feat:<35} {coef:.4f}")

## 7. Correlation Matrix

In [ ]:
corr = correlation_matrix(df)
corr

## 8. Visualizations

### 8.1 Distribution of Departure Delays

In [ ]:
from visualizations import (
    plot_delay_distribution, plot_delay_by_transport,
    plot_peak_vs_offpeak, plot_delay_by_congestion,
    plot_delay_category_breakdown, plot_correlation_heatmap,
    plot_feature_importance,
)

plot_delay_distribution(df)
plt.close("all")

# Display inline
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df["actual_departure_delay_min"], bins=25,
        edgecolor="white", color="#4C72B0", alpha=0.85)
mean_val = df["actual_departure_delay_min"].mean()
ax.axvline(mean_val, color="red", linestyle="--",
           label=f"Mean = {mean_val:.1f} min")
ax.set_title("Distribution of Departure Delays")
ax.set_xlabel("Delay (minutes)")
ax.set_ylabel("Number of Trips")
ax.legend()
plt.show()

### 8.2 Average Delay by Transport Type

In [ ]:
plot_delay_by_transport(df)
plt.close("all")

means = df.groupby("transport_type")["actual_departure_delay_min"].mean().sort_values()
colors = ["#55A868", "#4C72B0", "#C44E52", "#8172B2"]
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(means.index, means.values,
              color=colors[:len(means)], edgecolor="white")
for bar, val in zip(bars, means.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f"{val:.1f}", ha="center", fontsize=10)
ax.set_title("Average Delay by Transport Type")
ax.set_ylabel("Avg Delay (min)")
plt.show()

### 8.3 Peak vs Off-Peak Delay Distribution

In [ ]:
plot_peak_vs_offpeak(df)
plt.close("all")

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df[df["peak_hour"] == 1]["actual_departure_delay_min"],
        bins=20, alpha=0.6, label="Peak Hour", color="#C44E52")
ax.hist(df[df["peak_hour"] == 0]["actual_departure_delay_min"],
        bins=20, alpha=0.6, label="Off-Peak", color="#4C72B0")
ax.set_title("Peak vs Off-Peak Delay Distribution")
ax.set_xlabel("Delay (minutes)")
ax.set_ylabel("Frequency")
ax.legend()
plt.show()

### 8.4 Delay Spread by Traffic Congestion

In [ ]:
plot_delay_by_congestion(df)
plt.close("all")

df_tmp = df.copy()
df_tmp["congestion_level"] = pd.cut(
    df_tmp["traffic_congestion_index"],
    bins=[0, 3, 6, 10],
    labels=["Low (0-3)", "Medium (3-6)", "High (6-10)"],
    include_lowest=True,
)
levels = ["Low (0-3)", "Medium (3-6)", "High (6-10)"]
data = [df_tmp[df_tmp["congestion_level"] == lv]["actual_departure_delay_min"].dropna().values
        for lv in levels]

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(data, labels=levels, patch_artist=True)
palette = ["#55A868", "#FFD700", "#C44E52"]
for patch, color in zip(bp["boxes"], palette):
    patch.set_facecolor(color)
ax.set_title("Delay Spread by Traffic Congestion Level")
ax.set_ylabel("Delay (minutes)")
ax.set_xlabel("Congestion Level")
plt.show()

### 8.5 Delay Category Breakdown by Transport Type

In [ ]:
plot_delay_category_breakdown(df)
plt.close("all")

cat_order = ["On Time", "Minor", "Moderate", "Severe"]
ct = pd.crosstab(df["transport_type"], df["delay_category"])
ct = ct.reindex(columns=cat_order, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 6))
ct.plot(kind="bar", stacked=True, ax=ax, edgecolor="white",
        color=["#55A868", "#FFD700", "#FF8C00", "#C44E52"])
ax.set_title("Delay Category Breakdown by Transport Type")
ax.set_ylabel("Number of Trips")
ax.set_xlabel("")
ax.legend(title="Category")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.show()

### 8.6 Feature Correlation Heatmap

In [ ]:
plot_correlation_heatmap(corr)
plt.close("all")

fig, ax = plt.subplots(figsize=(9, 7))
cax = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr.columns, fontsize=9)
ax.set_title("Feature Correlation Matrix")
for i in range(len(corr)):
    for j in range(len(corr)):
        val = corr.values[i, j]
        color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=8, color=color)
plt.show()

### 8.7 Logistic Regression — Feature Importance

In [ ]:
plot_feature_importance(importance)
plt.close("all")

sorted_items = sorted(importance.items(), key=lambda kv: kv[1], reverse=True)
names = [k for k, _ in sorted_items]
values = [v for _, v in sorted_items]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(names, values, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Absolute Coefficient")
ax.set_title("Logistic Regression — Feature Importance")
ax.invert_yaxis()
plt.show()

## 9. Conclusion

This analysis explored 2,000 public-transport trips and examined how factors such as
transport type, peak hours, traffic congestion, and weather conditions influence departure
delays.

**Key findings:**
- Delays are broadly distributed, with a notable right tail of severe delays.
- Traffic congestion is among the strongest predictors of delay.
- The logistic regression model provides a reasonable baseline for binary delay prediction.
- Rule-based thresholding identifies high-risk routes that could be targeted for schedule
  adjustments.